# Sprint 4 — Modelagem e avaliação

Este notebook contém a implementação completa: leitura e limpeza, EDA, criação das variáveis, divisão temporal, baselines, ajuste de hiperparâmetros, avaliação por subgrupos e gravação dos três modelos em `models/`. Ele é autocontido e não depende de um arquivo Python auxiliar.

> **Limitação essencial:** a base disponível contém valor e data de inscrição, mas não resultados observados de pagamento, inadimplência ou recuperação. Os alvos abaixo são **proxies sintéticos**, criados por regras explícitas para demonstrar o pipeline. As métricas medem aderência a esses rótulos sintéticos, não desempenho no mundo real. Não usar os escores para decidir cobrança real. Para uma previsão válida, é necessário integrar históricos observados de pagamento/baixa e construir coortes temporais.

A coluna `Nome` é descartada antes da modelagem e não é usada nas features nem salva nos artefatos. As únicas features são `log_valor_divida` e `idade_divida_dias`.

## 1. Configuração e funções do pipeline

A célula abaixo reúne as funções usadas nas etapas seguintes. O notebook procura o CSV na pasta atual ou em uma pasta acima, para funcionar aberto a partir da raiz do projeto ou de `notebooks/`.

In [ ]:
from IPython.display import display
"""Treina artefatos demonstrativos do Sprint 4 com rótulos sintéticos.

O CSV usado pelo projeto não possui resultados observados de pagamento,
inadimplência ou recuperação. Os alvos gerados aqui são proxies didáticos e
não podem ser interpretados como previsões para uso operacional.
"""

import platform
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

CURRENT = Path.cwd()
ROOT = next(
    (candidate for candidate in [CURRENT, *CURRENT.parents]
     if (candidate / "relatorio_dívida_ativa.csv").exists()),
    CURRENT,
)
DATA_PATH = ROOT / "relatorio_dívida_ativa.csv"
MODELS_DIR = ROOT / "models"
FEATURES = ["log_valor_divida", "idade_divida_dias"]
SEED = 42
DATA_REFERENCIA = pd.Timestamp("2026-10-01")
FIM_TREINO = pd.Timestamp("2026-04-30")
FIM_VALIDACAO = pd.Timestamp("2026-06-30")


def metricas_binarias(y_true: pd.Series, y_pred: np.ndarray) -> dict:
    """Calcula métricas binárias e matriz de confusão para a classe positiva 1."""
    return {
        "acuracia": float(accuracy_score(y_true, y_pred)),
        "precisao": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "matriz_confusao_0_1": confusion_matrix(
            y_true, y_pred, labels=[0, 1]
        ).tolist(),
    }


def carregar_e_preparar(caminho: Path = DATA_PATH) -> pd.DataFrame:
    """Lê a dívida ativa e mantém somente campos necessários à demonstração."""
    raw = pd.read_csv(caminho, sep=";", encoding="utf-8-sig")
    requeridas = {"Valor Total da Dívida", "Data da Inscrição"}
    faltantes = requeridas.difference(raw.columns)
    if faltantes:
        raise ValueError(f"Colunas ausentes na base: {sorted(faltantes)}")

    # Remove duplicatas reais antes de descartar Nome; registros distintos com
    # mesmo valor e data continuam contando como observações distintas.
    raw = raw.drop_duplicates()
    # A coluna Nome e quaisquer outros identificadores são descartados.
    df = raw[["Valor Total da Dívida", "Data da Inscrição"]].copy()
    df["valor_divida"] = pd.to_numeric(
        df["Valor Total da Dívida"]
        .astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )
    df["data_inscricao"] = pd.to_datetime(
        df["Data da Inscrição"], format="%d/%m/%Y", errors="coerce"
    )
    df = (
        df.dropna(subset=["valor_divida", "data_inscricao"])
        .loc[lambda x: x["valor_divida"] >= 0]
        .sort_values("data_inscricao")
        .reset_index(drop=True)
    )
    df["idade_divida_dias"] = (
        DATA_REFERENCIA - df["data_inscricao"]
    ).dt.days.clip(lower=0)
    df["log_valor_divida"] = np.log1p(df["valor_divida"])

    # Regra transparente de score sintético; não representa recuperação observada.
    score = (
        0.65 * np.exp(-(df["idade_divida_dias"] / 365.0) / 3.0)
        + 0.35 * np.exp(-df["log_valor_divida"] / 5.0)
    ).clip(0, 1)
    df["score_recuperabilidade_proxy"] = score
    df["faixa_recuperabilidade_proxy"] = pd.cut(
        score,
        bins=[-np.inf, 0.65, 0.75, np.inf],
        labels=["baixa", "media", "alta"],
        right=False,
    ).astype(str)

    # Bernoulli reproduzível com probabilidade criada a partir das mesmas features.
    logit = -3.8 + 0.35 * df["log_valor_divida"] + 0.35 * (
        df["idade_divida_dias"] / 365.0
    )
    prob = 1.0 / (1.0 + np.exp(-logit))
    df["inadimplencia_proxy"] = (
        np.random.default_rng(SEED).random(len(df)) < prob
    ).astype(int)
    return df


def _validar_particoes(df: pd.DataFrame) -> tuple[pd.Series, pd.Series, pd.Series]:
    train = df["data_inscricao"] <= FIM_TREINO
    validation = (df["data_inscricao"] > FIM_TREINO) & (
        df["data_inscricao"] <= FIM_VALIDACAO
    )
    test = df["data_inscricao"] > FIM_VALIDACAO
    if not train.any() or not validation.any() or not test.any():
        raise ValueError(
            "A divisão temporal gerou partição vazia. Revise os limites de data."
        )
    return train, validation, test


def run_training() -> dict:
    """Treina, avalia e salva baseline.joblib e os dois modelos finais."""
    if not DATA_PATH.exists():
        raise FileNotFoundError(f"Base de entrada não encontrada: {DATA_PATH}")
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    df = carregar_e_preparar()
    train, validation, test = _validar_particoes(df)
    X = df[FEATURES].astype(float)
    y = df["inadimplencia_proxy"]
    X_train, y_train = X.loc[train], y.loc[train]
    X_val, y_val = X.loc[validation], y.loc[validation]
    X_test, y_test = X.loc[test], y.loc[test]

    # Baselines especificados na tarefa: Regressão Logística e Árvore de Decisão.
    baselines = {
        "regressao_logistica": make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=1000, class_weight="balanced", random_state=SEED
            ),
        ),
        "arvore_decisao": DecisionTreeClassifier(
            max_depth=6,
            min_samples_leaf=50,
            class_weight="balanced",
            random_state=SEED,
        ),
    }
    baseline_validacao = {}
    for nome, modelo in baselines.items():
        modelo.fit(X_train, y_train)
        baseline_validacao[nome] = metricas_binarias(y_val, modelo.predict(X_val))

    # Ajuste de hiperparâmetros com CV temporal somente dentro do treino.
    busca = GridSearchCV(
        RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1,
        ),
        param_grid={"max_depth": [8, None], "min_samples_leaf": [10, 30]},
        scoring="f1",
        cv=TimeSeriesSplit(n_splits=3),
        n_jobs=-1,
        refit=True,
    )
    busca.fit(X_train, y_train)
    inadimplencia = busca.best_estimator_
    validacao_inadimplencia = metricas_binarias(
        y_val, inadimplencia.predict(X_val)
    )

    X_train_val = pd.concat([X_train, X_val]).sort_index()
    y_train_val = pd.concat([y_train, y_val]).sort_index()
    inadimplencia.fit(X_train_val, y_train_val)
    teste_inadimplencia = metricas_binarias(
        y_test, inadimplencia.predict(X_test)
    )
    for modelo in baselines.values():
        modelo.fit(X_train_val, y_train_val)

    # Classificador didático para reproduzir as três faixas do score proxy.
    y_rec = df["faixa_recuperabilidade_proxy"]
    recuperabilidade = RandomForestClassifier(
        n_estimators=160,
        max_depth=10,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    )
    recuperabilidade.fit(X_train, y_rec.loc[train])
    validacao_recuperabilidade = {
        "acuracia": float(
            accuracy_score(y_rec.loc[validation], recuperabilidade.predict(X_val))
        ),
        "f1_macro": float(
            f1_score(
                y_rec.loc[validation], recuperabilidade.predict(X_val), average="macro"
            )
        ),
    }
    recuperabilidade.fit(X_train_val, y_rec.loc[train | validation])
    pred_rec = recuperabilidade.predict(X_test)
    teste_recuperabilidade = {
        "acuracia": float(accuracy_score(y_rec.loc[test], pred_rec)),
        "f1_macro": float(
            f1_score(y_rec.loc[test], pred_rec, average="macro")
        ),
        "matriz_confusao_baixa_media_alta": confusion_matrix(
            y_rec.loc[test], pred_rec, labels=["baixa", "media", "alta"]
        ).tolist(),
    }

    # Diagnóstico de métricas por subgrupos, guardado como base para Sprint 5.
    test_frame = df.loc[
        test, ["valor_divida", "idade_divida_dias", "inadimplencia_proxy"]
    ].copy()
    test_frame["predito_proxy"] = inadimplencia.predict(X_test)
    test_frame["faixa_valor"] = pd.cut(
        test_frame["valor_divida"],
        [-np.inf, 500, 5000, np.inf],
        labels=["ate_500", "500_a_5000", "acima_5000"],
    )
    test_frame["faixa_idade"] = pd.cut(
        test_frame["idade_divida_dias"],
        [-np.inf, 30, 60, np.inf],
        labels=["ate_30_dias", "31_a_60_dias", "acima_60_dias"],
    )
    subgroup_metrics = []
    for variable in ["faixa_valor", "faixa_idade"]:
        for group, block in test_frame.groupby(variable, observed=True):
            subgroup_metrics.append(
                {
                    "variavel": variable,
                    "grupo": str(group),
                    "n": int(len(block)),
                    **metricas_binarias(
                        block["inadimplencia_proxy"], block["predito_proxy"]
                    ),
                }
            )

    metadata = {
        "features": FEATURES,
        "data_referencia": DATA_REFERENCIA.date().isoformat(),
        "limite_treino": FIM_TREINO.date().isoformat(),
        "limite_validacao": FIM_VALIDACAO.date().isoformat(),
        "target_status": "sintetico_proxy_nao_observado",
        "aviso": (
            "Não usar para decisões reais: a base não contém rótulos observados "
            "de pagamento, inadimplência ou recuperação."
        ),
        "seed": SEED,
        "versions": {
            "python": platform.python_version(),
            "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__,
        },
    }
    joblib.dump(
        {**metadata, "estimators": baselines, "validation_metrics": baseline_validacao},
        MODELS_DIR / "baseline.joblib",
    )
    joblib.dump(
        {
            **metadata,
            "estimator": inadimplencia,
            "best_params": busca.best_params_,
            "cv_f1": float(busca.best_score_),
            "validation_metrics": validacao_inadimplencia,
            "test_metrics_proxy": teste_inadimplencia,
        },
        MODELS_DIR / "inadimplencia.joblib",
    )
    joblib.dump(
        {
            **metadata,
            "estimator": recuperabilidade,
            "classes": list(recuperabilidade.classes_),
            "validation_metrics": validacao_recuperabilidade,
            "test_metrics_proxy": teste_recuperabilidade,
        },
        MODELS_DIR / "recuperabilidade.joblib",
    )

    report = {
        "input": DATA_PATH.name,
        "records_after_cleaning": int(len(df)),
        "date_range": [df.data_inscricao.min().date().isoformat(), df.data_inscricao.max().date().isoformat()],
        "partition_sizes": {
            "train": int(train.sum()),
            "validation": int(validation.sum()),
            "test": int(test.sum()),
        },
        "proxy_label_counts": {
            str(key): int(value)
            for key, value in df["inadimplencia_proxy"].value_counts().sort_index().items()
        },
        "baseline_validation": baseline_validacao,
        "inadimplencia_best_params": busca.best_params_,
        "inadimplencia_cv_f1": float(busca.best_score_),
        "inadimplencia_validation": validacao_inadimplencia,
        "inadimplencia_test_proxy": teste_inadimplencia,
        "recuperabilidade_validation": validacao_recuperabilidade,
        "recuperabilidade_test_proxy": teste_recuperabilidade,
        "subgroup_metrics_proxy": subgroup_metrics,
        "note": metadata["aviso"],
        "artifacts": ["models/baseline.joblib", "models/inadimplencia.joblib", "models/recuperabilidade.joblib"],
    }
    return report



## 2. Carregamento, limpeza e EDA resumida

As duplicatas exatas são removidas antes de descartar nomes, para não fundir dívidas distintas que tenham o mesmo valor e data. A moeda é convertida do padrão brasileiro e a idade usa a data de referência definida no código.

In [ ]:
from IPython.display import display
df = carregar_e_preparar()
print(f"Registros válidos após limpeza: {len(df):,}")
print(f"Período de inscrição: {df.data_inscricao.min().date()} a {df.data_inscricao.max().date()}")
display(df[["valor_divida", "idade_divida_dias", "log_valor_divida"]].describe().round(2))
print("Distribuição das faixas sintéticas de recuperabilidade:")
display(df["faixa_recuperabilidade_proxy"].value_counts().rename_axis("faixa").to_frame("registros"))

## 3. Alvos sintéticos e split temporal

`inadimplencia_proxy` é uma simulação com semente fixa, baseada em valor e idade. As faixas de recuperabilidade são calculadas por uma regra explícita (baixa < 0,65; média de 0,65 a < 0,75; alta ≥ 0,75). Nenhum desses rótulos veio de um resultado real.

A separação cronológica é: treino até 30/04/2026, validação de 01/05 a 30/06/2026 e teste após 30/06/2026. A busca de hiperparâmetros ocorre apenas no treino com `TimeSeriesSplit`.

In [ ]:
train_mask, validation_mask, test_mask = _validar_particoes(df)
for name, mask in [("treino", train_mask), ("validação", validation_mask), ("teste", test_mask)]:
    print(f"{name}: {int(mask.sum()):,} linhas")
print("\nBaselines: Regressão Logística e Árvore de Decisão.")
print("Modelo final de inadimplência proxy: Random Forest com busca temporal de hiperparâmetros.")
print("Modelo de recuperabilidade proxy: Random Forest multiclasse.")

## 4. Treino, métricas e serialização

Esta célula treina os baselines, ajusta o classificador de inadimplência, treina o modelo de faixas de recuperabilidade, calcula métricas no teste temporal e por subgrupo e salva `baseline.joblib`, `inadimplencia.joblib` e `recuperabilidade.joblib` na pasta `models/` da raiz do projeto.

In [ ]:
relatorio = run_training()
print("Tamanhos das partições:", relatorio["partition_sizes"])
print("Baselines — validação:")
display(pd.DataFrame(relatorio["baseline_validation"]).T.round(4))
print("Inadimplência proxy — teste temporal:")
display(pd.DataFrame([relatorio["inadimplencia_test_proxy"]]).drop(columns="matriz_confusao_0_1").round(4))
print("Recuperabilidade proxy — teste temporal:")
display(pd.DataFrame([relatorio["recuperabilidade_test_proxy"]]).drop(columns="matriz_confusao_baixa_media_alta").round(4))
print("Métricas por subgrupo (rótulos sintéticos):")
display(pd.DataFrame(relatorio["subgroup_metrics_proxy"]).drop(columns="matriz_confusao_0_1").round(4))
print("\nModelos gravados:")
for name in ["baseline.joblib", "inadimplencia.joblib", "recuperabilidade.joblib"]:
    path = MODELS_DIR / name
    print(f"{path.relative_to(ROOT)} — {path.stat().st_size:,} bytes")

## Conclusão e próximos passos

A trilha é reproduzível para demonstração acadêmica. As métricas do teste e dos subgrupos usam alvos sintéticos e não devem ser apresentadas como acurácia real de inadimplência. Para avançar, a equipe precisa substituir os proxies por desfechos observados, documentar as fontes e janelas de observação e reavaliar a separação temporal antes de qualquer uso operacional.